In [1]:
import pandas as pd
import numpy as np

np.random.seed(23)

### Building the dataset
- It has 3 features
- It is taken from a zero-mean and 1,1,1 mean distribution at random

In [ ]:
mu_vec1 = np.array([0, 0, 0])  # zero mean
cov_mat1 = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]])
class1_sample = np.random.multivariate_normal(mu_vec1, cov_mat1, 20)

In [ ]:
df = pd.DataFrame(class1_sample, columns=["feature1", "feature2", "feature3"])
df["target"] = 1
df.head()

,feature1,feature2,feature3,target
0,0.666988,0.025813,-0.777619,1
1,0.948634,0.701672,-1.051082,1
2,-0.367548,-1.137460,-1.322148,1
3,1.772258,-0.347459,0.670140,1
4,0.322272,0.060343,-1.043450,1


In [ ]:
mu_vec2 = np.array([1, 1, 1])  # one mean sample
cov_mat2 = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]])
class2_sample = np.random.multivariate_normal(mu_vec2, cov_mat2, 20)

df2 = pd.DataFrame(class2_sample, columns=["feature1", "feature2", "feature3"])
df2["target"] = 0

df = pd.concat([df, df2], ignore_index=True)
df = df.sample(40)
df.head()

,feature1,feature2,feature3,target
2,-0.367548,-1.137460,-1.322148,1
34,0.177061,-0.598109,1.226512,0
14,0.420623,0.411620,-0.071324,1
11,1.968435,-0.547788,-0.679418,1
12,-2.506230,0.146960,0.606195,1


---

In [10]:
import plotly.express as px

fig = px.scatter_3d(
    df,
    x=df["feature1"],
    y=df["feature2"],
    z=df["feature3"],
    color=df["target"].astype("str"),
)

fig.update_traces(
    marker=dict(size=12, line=dict(width=2, color="DarkSlateGrey")),
    selector=dict(mode="markers"),
)
fig.show()

In [15]:
# Step 1 - Apply Standard Scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df.iloc[:, 0:3] = scaler.fit_transform(df.iloc[:, 0:3])

In [16]:
# Step 2 - Find Covariance Matrix

cov_matrix = np.cov([df.iloc[:, x] for x in range(0, 3)])
print("Covariance Matrix: \n", cov_matrix)

Covariance Matrix: 
 [[1.02564103 0.20478114 0.080118  ]
 [0.20478114 1.02564103 0.19838882]
 [0.080118   0.19838882 1.02564103]]


In [17]:
# Step 3: Finding Eigenvectors, Eigenvalues

e_vals, e_vecs = np.linalg.eig(cov_matrix)
e_vals

array([1.3536065 , 0.94557084, 0.77774573])

In [18]:
# Select 2 eigen vectors

# principal components
pc = e_vecs[0:2]
pc

array([[-0.53875915, -0.69363291,  0.47813384],
       [-0.65608325, -0.01057596, -0.75461442]])

In [20]:
transformed_df = np.dot(df.iloc[:, 0:3], pc.T)
transformed_df[0:5]

array([[ 0.59943321,  1.79586208],
       [ 1.05691919, -0.2127375 ],
       [-0.27187555,  0.49822203],
       [-0.62158585,  0.02311035],
       [ 1.56728555,  1.73096695]])

In [ ]:
new_df = pd.DataFrame(transformed_df, columns=["PC1", "PC2"])
new_df["target"] = df["target"].values
new_df.head()

,PC1,PC2,target
0,0.599433,1.795862,1
1,1.056919,-0.212737,0
2,-0.271876,0.498222,1
3,-0.621586,0.023110,1
4,1.567286,1.730967,1


In [22]:
new_df["target"] = new_df["target"].astype("str")
fig = px.scatter(
    x=new_df["PC1"],
    y=new_df["PC2"],
    color=new_df["target"],
    color_discrete_sequence=px.colors.qualitative.G10,
)

fig.update_traces(
    marker=dict(size=12, line=dict(width=2, color="DarkSlateGrey")),
    selector=dict(mode="markers"),
)
fig.show()